# Nemotron 3.5 + NeMo: prueba inicial en Colab

Esta notebook valida primero el modelo y el runtime, **sin placa ni servidor live**. Monta Google Drive, encuentra los tres audios cortos, los convierte a WAV mono de 16 kHz y ejecuta:

1. transcripción offline con el checkpoint oficial;
2. simulación streaming cache-aware oficial de NeMo a 320 ms.

Los resultados y la versión exacta de NeMo quedan guardados en Drive. Ejecutá `Runtime -> Run all` con una GPU T4 o mejor.

Modelo: https://huggingface.co/nvidia/nemotron-3.5-asr-streaming-0.6b

## 1. Verificar GPU

In [ ]:
import os, platform, subprocess, sys
import torch
print('python :', platform.python_version())
print('torch  :', torch.__version__)
print('cuda   :', torch.version.cuda)
print('gpu    :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'Activá Runtime -> Change runtime type -> GPU antes de continuar'
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'], check=True)

## 2. Montar Drive y configurar caches

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

# Cambiá solamente estas rutas si tu Drive usa otro nombre. AUDIO_DIR puede
# quedar en None: más abajo se buscan automáticamente las ubicaciones usadas
# por los experimentos anteriores.
DRIVE_BASE = Path('/content/drive/MyDrive/Tesis-subtitles')
AUDIO_DIR = None
NEMOTRON_DRIVE = DRIVE_BASE / 'nemotron'
RESULTS_ROOT = NEMOTRON_DRIVE / 'results'
CACHE_ROOT = NEMOTRON_DRIVE / 'cache'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

# Los pesos se descargan una vez y se reutilizan desde Drive.
os.environ['HF_HOME'] = str(CACHE_ROOT / 'huggingface')
os.environ['NEMO_CACHE_DIR'] = str(CACHE_ROOT / 'nemo')
os.environ['TORCH_HOME'] = str(CACHE_ROOT / 'torch')

# El modelo es público, pero un HF_TOKEN evita límites de descarga. Agregalo
# como Secret de Colab si ya tenés uno; nunca se imprime ni se guarda.
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
print('HF auth:', 'configured' if hf_token else 'anonymous (public model)')
print('results:', RESULTS_ROOT)
print('cache  :', CACHE_ROOT)

## 3. Traer `dev/direct-connect` y NeMo

La rama debe estar pusheada a GitHub. NeMo se clona desde su repositorio oficial y el SHA resuelto se guarda en el resultado; cuando confirmemos una combinación estable, ese SHA se fijará permanentemente.

In [ ]:
REPO_URL = 'https://github.com/Nacholazabal/subtitle_overlay_fw.git'
REPO_BRANCH = 'dev/direct-connect'
REPO_DIR = Path('/content/subtitle_overlay_fw')
NEMO_REPO = 'https://github.com/NVIDIA-NeMo/NeMo.git'
NEMO_REF = 'main'
NEMO_DIR = Path('/content/NeMo')

remote = subprocess.run(['git', 'ls-remote', '--heads', REPO_URL, REPO_BRANCH], capture_output=True, text=True)
assert remote.stdout.strip(), f'No existe {REPO_BRANCH} en GitHub. Desde WSL: git push -u origin {REPO_BRANCH}'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'switch', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)

if not NEMO_DIR.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', '--depth', '1', '--branch', NEMO_REF, NEMO_REPO, str(NEMO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(NEMO_DIR), 'pull', '--ff-only'], check=True)

PROJECT_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
NEMO_COMMIT = subprocess.check_output(['git', '-C', str(NEMO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
print('project commit:', PROJECT_COMMIT)
print('NeMo commit   :', NEMO_COMMIT)

## 4. Instalar NeMo Speech

In [ ]:
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'ffmpeg', 'libsndfile1'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'Cython', 'packaging'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{NEMO_DIR}[asr]'], check=True)

# pip editable registra el checkout mediante un .pth que normalmente se lee
# recién al iniciar Python. Este kernel ya estaba vivo, así que agregamos el
# checkout explícitamente y comprobamos el import AHORA, no varias celdas después.
import importlib
assert (NEMO_DIR / 'nemo').is_dir(), f'el checkout no contiene el paquete nemo: {NEMO_DIR}'
if str(NEMO_DIR) not in sys.path:
    sys.path.insert(0, str(NEMO_DIR))
importlib.invalidate_caches()
import nemo
import nemo.collections.asr as _nemo_asr_smoke
print('NeMo Speech installed + importable from', NEMO_COMMIT)
print('nemo module:', nemo.__file__)

## 5. Encontrar y preparar los audios de Drive

In [ ]:
# Verificar primero el checkout físico. Si esto falla, la rama de Colab está
# desactualizada; si pasa, cualquier error posterior es sólo cache/import path.
probe_module = REPO_DIR / 'server' / 'evaluation' / 'probe.py'
package_marker = REPO_DIR / 'server' / '__init__.py'
assert probe_module.is_file() and package_marker.is_file(), (
    f'La copia de Colab no contiene {probe_module}. ' 
    f'Commit actual: {PROJECT_COMMIT}. Reejecutá la celda 3.'
)

# Colab puede conservar un paquete ajeno o una versión anterior llamado
# `server`. Poner el repo primero no alcanza si ese módulo ya está cacheado.
repo_path = str(REPO_DIR)
sys.path[:] = [path for path in sys.path if path != repo_path]
sys.path.insert(0, repo_path)
for module_name in [
    name for name in list(sys.modules)
    if name == 'server' or name.startswith('server.')
]:
    del sys.modules[module_name]
import importlib
importlib.invalidate_caches()
from server.evaluation.probe import (
    NemotronProbeConfig, build_offline_transcribe_config,
    prepare_probe_inputs, select_drive_audio_dir, write_json
)
import server as project_server
assert Path(project_server.__file__).resolve() == package_marker.resolve(), (
    f'Se importó otro paquete server desde {project_server.__file__}'
)
print('project package:', project_server.__file__)
print('project helpers:', probe_module)

candidates = ([Path(AUDIO_DIR)] if AUDIO_DIR else []) + [
    DRIVE_BASE / 'stt-bench' / 'audio',
    DRIVE_BASE / 'stt_audio',
    DRIVE_BASE / 'nemotron' / 'audio',
    Path('/content/drive/MyDrive/Tesis-subtitles/stt_audio'),
]
audio_dir = select_drive_audio_dir(candidates)
config = NemotronProbeConfig(target_lang='es-ES', latency_ms=320)
WORK_DIR = Path('/content/nemotron-probe')
records, MANIFEST_PATH = prepare_probe_inputs(audio_dir, WORK_DIR)
print('audio dir:', audio_dir)
print('manifest :', MANIFEST_PATH)
for record in records:
    print(Path(record['audio_filepath']).name, record['duration'], 's', record['reference_kind'])

## 6. Cargar Nemotron y hacer la prueba offline

La primera ejecución descarga aproximadamente 2.4 GB. No cierres la celda mientras esté descargando.

In [ ]:
import importlib.metadata, json, time
from datetime import datetime, timezone
import nemo.collections.asr as nemo_asr

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
RUN_DIR = RESULTS_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)

load_started = time.perf_counter()
asr_model = nemo_asr.models.ASRModel.from_pretrained(model_name=config.model_id)
assert hasattr(asr_model, 'set_inference_prompt'), 'Checkpoint cargado sin language prompt support'
asr_model.set_inference_prompt(config.target_lang)
asr_model.encoder.set_default_att_context_size(att_context_size=list(config.att_context_size))
asr_model = asr_model.to('cuda').eval()
load_sec = time.perf_counter() - load_started
print('model class:', type(asr_model).__name__)
print('load sec   :', round(load_sec, 2))
print('context    :', config.att_context_size, '=', config.latency_ms, 'ms')

wav_paths = [record['audio_filepath'] for record in records]
# La ruta Lhotse de esta versión pierde el idioma al convertir una lista de
# paths en manifest temporal (prompt=None). El config no-Lhotse usa el fallback
# dinámico oficial del modelo con target_lang='es-ES'.
offline_config = build_offline_transcribe_config(asr_model, config)
infer_started = time.perf_counter()
with torch.inference_mode():
    hypotheses = asr_model.transcribe(audio=wav_paths, override_config=offline_config)
offline_sec = time.perf_counter() - infer_started
offline = []
for source, hypothesis in zip(records, hypotheses):
    text = str(getattr(hypothesis, 'text', hypothesis)).strip()
    item = {'audio': Path(source['source_audio_filepath']).name, 'text': text, 'duration_sec': source['duration']}
    offline.append(item)
    print('\n', item['audio'], '=>', text)

provenance = {
    'run_id': RUN_ID, 'project_commit': PROJECT_COMMIT, 'nemo_commit': NEMO_COMMIT,
    'nemo_toolkit_version': importlib.metadata.version('nemo_toolkit'),
    'torch_version': torch.__version__, 'cuda_version': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0), 'config': config.as_dict(),
    'model_load_sec': round(load_sec, 3), 'offline_inference_sec': round(offline_sec, 3),
}
write_json(RUN_DIR / 'provenance.json', provenance)
write_json(RUN_DIR / 'offline.json', {'reference_kind': 'model_output_not_human_truth', 'items': offline})
print('saved:', RUN_DIR)

## 7. Ejecutar la simulación streaming cache-aware oficial

Se libera el modelo offline para que el script oficial pueda cargar su propia instancia sin agotar la T4. `debug_mode=true` permite comprobar que aparecen hipótesis incrementales.

In [ ]:
import gc
del hypotheses, asr_model
gc.collect()
torch.cuda.empty_cache()

from server.evaluation.probe import build_official_streaming_command, parse_streaming_debug_transcripts
stream_dir = RUN_DIR / 'streaming'
stream_dir.mkdir(parents=True, exist_ok=True)
command = build_official_streaming_command(NEMO_DIR, MANIFEST_PATH, stream_dir, config)
print('running official NeMo cache-aware script...')
stream_started = time.perf_counter()
process = subprocess.run(command, capture_output=True, text=True, env=os.environ.copy())
stream_sec = time.perf_counter() - stream_started
full_log = process.stdout + '\n' + process.stderr
(RUN_DIR / 'streaming.log').write_text(full_log, encoding='utf-8')
print('\n'.join(full_log.splitlines()[-120:]))
if process.returncode != 0:
    raise RuntimeError(f'NeMo streaming probe failed with code {process.returncode}; log: {RUN_DIR / "streaming.log"}')

parsed = parse_streaming_debug_transcripts(full_log)
summary = {
    'elapsed_sec': round(stream_sec, 3),
    'partial_updates_logged': len(parsed['partials']),
    'final_batches_logged': len(parsed['finals']),
    **parsed,
}
write_json(RUN_DIR / 'streaming-summary.json', summary)
print('\nSTREAMING PROBE OK')
print('elapsed         :', round(stream_sec, 2), 's')
print('partial updates :', len(parsed['partials']))
print('final batches   :', len(parsed['finals']))
print('results        :', RUN_DIR)

## Qué enviar después

Cuando termine, copiá acá:

- las últimas líneas de la celda 7;
- `provenance.json`;
- `offline.json`;
- `streaming-summary.json`.

Con eso decidimos si la instalación y el RTF justifican construir `server/runtime/app.py` y conectarlo al bridge/firmware. Los `.txt` ausentes no bloquean esta prueba y no se reporta WER real.